# Exploratory Data Analysis, Modeling, and Verification

This notebook houses the statistical exploration, predictive modeling, and graphic mapping layers of the project. It reads the clean tables generated from the pipeline notebooks, combines regional records across shared administrative boundaries, fits Ordinary Least Squares (OLS) and instance-space estimators, and exports diagnostic graphics.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titlesize"] = 14

## Part 1: Dataset Compilation and Boundary Cross-Mapping

In [ ]:
print("Loading processed datasets...")
stations_df = pd.read_csv("data/processed/cleaned_alt_fuel_stations.csv")
ev_reg_df = pd.read_csv("data/processed/clean_EV_registrations.csv")
emissions_df = pd.read_csv("data/processed/transportation_CO2_emissions.csv")

state_ports = stations_df.groupby("State")["EVSE Total Total"].sum().reset_index()
state_ports.columns = ["State", "Total_Ports"]

merged_metrics = pd.merge(state_ports, ev_reg_df, on="State", how="inner")
recent_emissions = emissions_df[emissions_df["year"].astype(int) == 2024].copy()

state_mapping = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "California",
    "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia",
    "HI": "Hawaii", "ID": "Idaho", "IL": "Illinois", "IN": "Indiana", "IA": "Iowa",
    "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi", "MO": "Missouri",
    "MT": "Montana", "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NY": "New York", "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio",
    "OK": "Oklahoma", "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah", "VT": "Vermont",
    "VA": "Virginia", "WA": "Washington", "WV": "West Virginia", "WI": "Wisconsin", "WY": "Wyoming"
}
merged_metrics["State_Full"] = merged_metrics["State"].map(state_mapping)

final_analysis_df = pd.merge(
    merged_metrics, 
    recent_emissions[["stateId", "value"]],
    left_on="State_Full", 
    right_on="stateId", 
    how="inner"
).rename(columns={"value": "Transportation_CO2_MMT"})

print(f"✓ Consolidated workspace generated with {len(final_analysis_df)} active state matrices.")

## Part 2: Linear Modeling (Infrastructure Density vs. Local Procurement Volume)

In [ ]:
X_linear = final_analysis_df[["Total_Ports"]].values
y_linear = final_analysis_df["EV_Registrations"].values

linear_model = LinearRegression()
linear_model.fit(X_linear, y_linear)

r2_lin = r2_score(y_linear, linear_model.predict(X_linear))
slope_coeff = linear_model.coef_[0]

print(f"Ordinary Least Squares R² Score: {r2_lin:.3f}")
print(f"Calculated Port Coefficient Vector (Slope): {slope_coeff:.2f}")

## Part 3: Instance-Space Modeling (Fleet Scaling vs. Macro Sector Emissions)

In [ ]:
X_multi = final_analysis_df[["Total_Ports", "EV_Registrations"]].values
y_emissions = final_analysis_df["Transportation_CO2_MMT"].values

X_train, X_test, y_train, y_test = train_test_split(X_multi, y_emissions, test_size=0.2, random_state=42)

knn_model = KNeighborsRegressor(n_neighbors=3)
knn_model.fit(X_train, y_train)

knn_predictions = knn_model.predict(X_test)
r2_knn = r2_score(y_test, knn_predictions)
rmse_knn = np.sqrt(mean_squared_error(y_test, knn_predictions))

print(f"KNN Test Partition R² Score: {r2_knn:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse_knn:.2f} MMT")

## Part 4: Graphic Export Routines

In [ ]:
os.makedirs("plots", exist_ok=True)

# Visualization 1: OLS Relationship Map
plt.figure()
sns.regplot(data=final_analysis_df, x="Total_Ports", y="EV_Registrations", color="teal", scatter_kws={"alpha": 0.6})
plt.title("State EV Procurement Scales as a Function of Public Charging Ports")
plt.xlabel("Total Public Charging Ports Available")
plt.ylabel("Registered Consumer Electric Vehicles")
plt.tight_layout()
plt.savefig("plots/linear_infrastructure_vs_adoption.png", dpi=300)
plt.close()

# Visualization 2: Multi-Variable Emission Scatter Grid
plt.figure()
sns.scatterplot(
    data=final_analysis_df, 
    x="EV_Registrations", 
    y="Transportation_CO2_MMT", 
    size="Total_Ports", 
    hue="Transportation_CO2_MMT",
    palette="viridis",
    sizes=(40, 400),
    alpha=0.8
)
plt.title("Regional Fleet Scaling Plotted Against Annual Transportation Sector Carbon Outputs")
plt.xlabel("Total Consumer EV Registrations")
plt.ylabel("Sector Carbon Footprint (Million Metric Tons CO2)")
plt.legend(title="Public Port Density", loc="upper left")
plt.tight_layout()
plt.savefig("plots/emissions_vs_fleet_scale.png", dpi=300)
plt.close()

print("✓ All diagnostic project charts saved to the local 'plots/' subfolder.")